<a href="https://colab.research.google.com/github/IbHansen/wb-debt-simulation/blob/main/test_interactive/test_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#@title Notebook setup

#This is code to manage dependencies if the notebook is executed in the google colab cloud service
import importlib.util
from IPython.display import HTML, display

ip = get_ipython()
is_colab = ip is not None and 'google.colab' in str(ip)

if is_colab:
    if importlib.util.find_spec("modelclass") is None:
        import sys
        !apt-get -qq install -y graphviz
        !{sys.executable} -m pip -qqq install ModelFlowIb > /dev/null 2>&1
        !mkdir -p data && curl -L https://raw.githubusercontent.com/IbHansen/wb-debt-simulation/main/test_interactive/data/idn.pcim -o data/idn.pcim


In [2]:
# -*- coding: utf-8 -*-
"""
gridwidget
==========

A lightweight editable DataFrame grid built entirely from plain ipywidgets
(no ipydatagrid dependency). Drop-in replacement for ``sheetwidget`` within
the ``modelwidget_input`` WidgetABC framework.

Usage
-----
::

    griddef = ['grid', {
        'heading': 'My adjustments',
        'content': {
            'update_col':   ['VAR_A', 'VAR_B'],
            'update_index': [2020, 2021, 2022],
            'operator':     '+',   # +, =, *, %
            'dec':          2,
        },
    }]

    w = make_widget(griddef)
    display(w.datawidget)

Or with an existing DataFrame::

    griddef = ['grid', {
        'heading': 'Edits',
        'content': {
            'update_df': my_df,
            'operator':  '+',
            'dec':       2,
        },
    }]
"""

from __future__ import annotations

from copy import deepcopy
from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List

import pandas as pd
from IPython.display import display
from ipywidgets import (
    FloatText,
    GridBox,
    HBox,
    HTML,
    Label,
    Layout,
    VBox,
)

# ---------------------------------------------------------------------------
# Import the framework from modelwidget_input
# ---------------------------------------------------------------------------
try:
    from modelwidget_input import SingleWidgetBase, register_widget
except ImportError:
    # Fallback: define minimal stubs so the file is self-contained
    from abc import ABC, abstractmethod

    class WidgetABC(ABC):
        @property
        @abstractmethod
        def datawidget(self): ...
        @abstractmethod
        def update_df(self, df, current_per): ...
        @abstractmethod
        def reset(self, g): ...

    class SingleWidgetBase(WidgetABC):
        def __init__(self, widgetdef):
            self.widgetdef = widgetdef
            self.content = widgetdef["content"]
            self.heading = widgetdef.get("heading", "Heading")

    def register_widget(cls):
        return cls


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _header_cell(text: str, width: str = "100px") -> HTML:
    return HTML(
        value=f"<b style='font-size:12px;'>{text}</b>",
        layout=Layout(
            width=width,
            min_width=width,
            display="flex",
            justify_content="center",
            align_items="center",
            padding="2px 4px",
            border="1px solid #ccc",
        ),
    )


def _corner_cell(width: str = "100px") -> HTML:
    return HTML(
        value="",
        layout=Layout(
            width=width,
            min_width=width,
            border="1px solid #ccc",
        ),
    )


# ---------------------------------------------------------------------------
# Widget
# ---------------------------------------------------------------------------

@register_widget
@dataclass
class gridwidget(SingleWidgetBase):
    """
    Editable DataFrame grid using only plain ipywidgets.

    Supports the same ``content`` keys as ``sheetwidget``:

    - ``update_col`` + ``update_index``  **or**  ``update_df``
    - ``operator`` (``+``, ``=``, ``*``, ``%``) – default ``+``
    - ``dec`` – decimal places shown (default 2)

    And the same ``widgetdef`` option:

    - ``transpose`` (bool, default True) – if True variables become rows
    """

    df_var: pd.DataFrame = field(init=False)
    org_df_var: pd.DataFrame = field(init=False)
    org_values: pd.DataFrame = field(init=False)

    trans: Callable[[str], str] = field(default=lambda x: x)
    transpose: bool = field(default=True)
    op: str = field(init=False, default="+")
    dec: int = field(init=False, default=2)

    _cells: Dict[tuple, FloatText] = field(init=False, default_factory=dict)
    _datawidget: Any = field(init=False, default=None)

    # ---- construction ------------------------------------------------------

    def __post_init__(self) -> None:
        super().__post_init__()
        self.op = self.content.get("operator", "+")
        self.dec = int(self.content.get("dec", 2))
        self.transpose = bool(self.widgetdef.get("transpose", True))

        # Build the backing DataFrame (same logic as sheetwidget)
        update_col = self.content.get("update_col")
        update_index = self.content.get("update_index")
        update_df = self.content.get("update_df")

        if update_col is not None and update_index is not None:
            cols = pd.Index(update_col) if not isinstance(update_col, pd.Index) else update_col
            idx = pd.Index(update_index) if not isinstance(update_index, pd.Index) else update_index
            self.df_var = pd.DataFrame(0.0, index=idx, columns=cols)
        elif isinstance(update_df, pd.DataFrame):
            self.df_var = update_df.copy()
        else:
            raise ValueError(
                "Provide either both 'update_col' and 'update_index', or 'update_df'."
            )

        renamed = self.df_var.copy().rename(columns=self.trans)
        self.org_df_var = renamed.T if self.transpose else renamed
        self.org_values = self.org_df_var.copy()

        # Build the grid UI
        self._build_grid()

    def _build_grid(self) -> None:
        df = self.org_df_var
        n_rows, n_cols = df.shape
        row_labels = [str(r) for r in df.index]
        col_labels = [str(c) for c in df.columns]

        # Sizing
        idx_width = f"{max(80, max((len(s) for s in row_labels), default=6) * 10 + 20)}px"
        cell_width = f"{max(80, max((len(s) for s in col_labels), default=6) * 10 + 20)}px"
        step = 10.0 ** (-self.dec)

        # Build cells row by row: header row, then data rows
        grid_children: list = []

        # Corner + column headers
        grid_children.append(_corner_cell(idx_width))
        for c in col_labels:
            grid_children.append(_header_cell(c, cell_width))

        # Data rows
        self._cells = {}
        for r_idx, r_label in enumerate(row_labels):
            grid_children.append(_header_cell(r_label, idx_width))
            for c_idx, c_label in enumerate(col_labels):
                val = float(df.iloc[r_idx, c_idx])
                ft = FloatText(
                    value=val,
                    step=step,
                    layout=Layout(width=cell_width, min_width=cell_width, padding="0px"),
                    style={"description_width": "0px"},
                )
                self._cells[(r_idx, c_idx)] = ft
                grid_children.append(ft)

        total_cols = n_cols + 1  # +1 for row header column
        template = " ".join([idx_width] + [cell_width] * n_cols)

        grid = GridBox(
            children=grid_children,
            layout=Layout(
                grid_template_columns=template,
                grid_gap="0px",
                overflow_x="auto",
                overflow_y="auto",
                max_height="400px",
                width="100%",
            ),
        )

        heading_label = Label(value=self.heading, layout=Layout(width="54%"))
        self._datawidget = VBox([heading_label, grid])

    # ---- WidgetABC interface -----------------------------------------------

    @property
    def datawidget(self) -> Any:
        return self._datawidget

    def _read_grid(self) -> pd.DataFrame:
        """Read current cell values back into a DataFrame shaped like org_df_var."""
        df = self.org_df_var.copy()
        for (r, c), ft in self._cells.items():
            df.iloc[r, c] = ft.value
        return df

    def update_df(self, df: pd.DataFrame, current_per: Any = None) -> None:
        updated = self._read_grid()

        if self.transpose:
            updated = updated.T

        updated.columns = self.df_var.columns
        updated.index = self.df_var.index
        df_slice = df.loc[updated.index, updated.columns].copy()

        match self.op:
            case "+":
                df.loc[updated.index, updated.columns] = df_slice + updated
            case "=":
                df.loc[updated.index, updated.columns] = updated
            case "*":
                df.loc[updated.index, updated.columns] = df_slice * updated
            case "%":
                df.loc[updated.index, updated.columns] = df_slice * (1 + updated / 100)
            case _:
                raise ValueError(
                    f"Unsupported operator {self.op!r} in gridwidget for {self.heading!r}."
                )

    def reset(self, g: Any) -> None:
        for (r, c), ft in self._cells.items():
            ft.value = float(self.org_values.iloc[r, c])

In [3]:
# -*- coding: utf-8 -*-
"""
gridwidget_example.py
=====================

Minimal example showing how to use ``gridwidget`` in a Jupyter notebook or
Google Colab.  Run each section in its own cell, or run the whole file.

No model or ipydatagrid required – just plain ipywidgets + pandas.
"""

import pandas as pd
from IPython.display import display

# ---- 1) Create a small DataFrame to edit -----------------------------------

baseline = pd.DataFrame(
    {
        "GDP":       [100.0, 102.0, 104.5],
        "INFLATION": [2.0,   2.3,   2.1],
        "UNEMP":     [5.0,   4.8,   4.6],
    },
    index=[2024, 2025, 2026],
)

print("=== Baseline ===")
display(baseline)

# ---- 2) Build the grid widget ----------------------------------------------

try:
    from gridwidget import gridwidget
except: 
    ...
    
    
widgetdef = {
    "heading": "Scenario shocks (additive)",
    "content": {
        "update_df": pd.DataFrame(0.0, index=baseline.index, columns=baseline.columns),
        "operator":  "+",
        "dec":       1,
    },
    "transpose": True,   # variables as rows, periods as columns
}

grid = gridwidget(widgetdef)
display(grid.datawidget)

# ---- 3) Apply edits to a copy of the baseline ------------------------------
#
# After changing some cell values in the grid above, run this cell to see the
# effect.

=== Baseline ===


,GDP,INFLATION,UNEMP
2024,100.0,2.0,5.0
2025,102.0,2.3,4.8
2026,104.5,2.1,4.6


In [4]:
experiment = baseline.copy()
grid.update_df(experiment)

print("\n=== Experiment (baseline + your edits) ===")
display(experiment)

# ---- 4) Reset the grid back to zeros ---------------------------------------
#
# grid.reset(None)


=== Experiment (baseline + your edits) ===


,GDP,INFLATION,UNEMP
2024,100.0,2.0,5.0
2025,102.0,2.3,4.8
2026,104.5,2.1,4.6
